In [1]:
import os

In [2]:
os.getcwd()

'/home/thienhb/Workspace/arxiv-paper-rag/notebooks'

In [3]:
os.chdir("..")

In [4]:
os.getcwd()

'/home/thienhb/Workspace/arxiv-paper-rag'

In [8]:
import requests
import subprocess
import time

In [10]:
def test_ollama_model(model_name, prompt, max_wait_time=60):
    """Test an Ollama model with a prompt."""
    print(f"Testing {model_name} with prompt: '{prompt}'")
    print("-" * 60)
    
    url = "http://localhost:11434/api/generate"
    data = {
        "model": model_name,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        print("Generating response (this may take 10-30 seconds)...")
        start_time = time.time()
        
        response = requests.post(url, json=data, timeout=max_wait_time)
        
        if response.status_code == 200:
            result = response.json()
            response_text = result.get('response', '').strip()
            
            elapsed_time = time.time() - start_time
            print(f"Response generated in {elapsed_time:.1f} seconds")
            print("\nRESPONSE:")
            print("=" * 40)
            print(response_text)
            print("=" * 40)
            
            if 'model' in result:
                print(f"\nModel: {result['model']}")
            if 'total_duration' in result:
                duration_ms = result['total_duration'] / 1000000
                print(f"Generation time: {duration_ms:.0f}ms")
                
            return True
            
        else:
            print(f"API error: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except requests.exceptions.ConnectionError:
        print("Could not connect to Ollama API")
        print("Make sure Ollama is running: docker compose ps")
        return False
    except requests.exceptions.Timeout:
        print("Request timed out")
        print("Model might be loading for the first time (this is normal)")
        return False
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False

test_prompt = "What is machine learning in one sentence?"
success = test_ollama_model("llama3.1:1b", test_prompt)

if success:
    print("\nSUCCESS! Your local AI model is working!")
    print("\nTry more prompts:")
    print('• test_ollama_model("llama3.1:1b", "Explain neural networks simply")')
    print('• test_ollama_model("llama3.1:1b", "Write a Python function to sort a list")')
else:
    print("\nTroubleshooting:")
    print("1. Verify the model is installed: docker exec rag-ollama ollama list")
    print("2. Check Ollama logs: docker compose logs ollama")
    print("3. Try again - first run takes longer to load model into memory")

Testing llama3.1:1b with prompt: 'What is machine learning in one sentence?'
------------------------------------------------------------
Generating response (this may take 10-30 seconds)...
API error: 404
Response: {"error":"model 'llama3.1:1b' not found"}

Troubleshooting:
1. Verify the model is installed: docker exec rag-ollama ollama list
2. Check Ollama logs: docker compose logs ollama
3. Try again - first run takes longer to load model into memory
